In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")

In [2]:
df = pd.read_csv("../data/paysim.csv")

df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df_fe = df.copy()

#sender balance change
df_fe["origin_balance_change"] = (
    df_fe["oldbalanceOrg"] -
    df_fe["newbalanceOrig"]
)

#Reciever Balance change
df_fe["destination_balance_change"] = (
    df_fe["newbalanceDest"] -
    df_fe["oldbalanceDest"]
)

In [4]:
#Percentage of balance removed
df_fe["origin_balance_removed_pct"] = np.where(
    df_fe["oldbalanceOrg"] > 0,
    df_fe["origin_balance_change"] / df_fe["oldbalanceOrg"],
    0
)

In [5]:
#Percentage increase in destination
df_fe["destination_balance_increase_pct"] = np.where(
    df_fe["oldbalanceDest"] > 0,
    df_fe["destination_balance_change"] / df_fe["oldbalanceDest"],
    0
)

In [6]:
#Check for balance difference

#sender
df_fe["origin_balance_error"] = (
    df_fe["oldbalanceOrg"]
    - df_fe["amount"]
    - df_fe["newbalanceOrig"]
)

#Reciever
df_fe["destination_balance_error"] = (
    df_fe["oldbalanceDest"]
    + df_fe["amount"]
    - df_fe["newbalanceDest"]
)

In [7]:
#High value transactions
threshold = df_fe["amount"].quantile(0.95)

df_fe["high_value_transaction"] = (
    df_fe["amount"] > threshold
).astype(int)

In [8]:
df_fe["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [9]:
#Transaction type encoding
df_fe = pd.get_dummies(
    df_fe,
    columns=["type"],
    drop_first=True,
    dtype=int
)

In [10]:
#Removal of customer ID
df_fe.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ],
    inplace=True
)

In [11]:
df_fe.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,origin_balance_change,destination_balance_change,origin_balance_removed_pct,destination_balance_increase_pct,origin_balance_error,destination_balance_error,high_value_transaction,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,0,9839.64,0.0,0.057834,0.0,0.0,9839.64,0,0,0,1,0
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,0,1864.28,0.0,0.087735,0.0,0.0,1864.28,0,0,0,1,0
2,1,181.00,181.0,0.00,0.0,0.0,1,0,181.00,0.0,1.000000,0.0,0.0,181.00,0,0,0,0,1
3,1,181.00,181.0,0.00,21182.0,0.0,1,0,181.00,-21182.0,1.000000,-1.0,0.0,21363.00,0,1,0,0,0
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,0,11668.14,0.0,0.280795,0.0,0.0,11668.14,0,0,0,1,0


In [14]:
df_fe.info()
df_fe.shape

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 19 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   step                              int64  
 1   amount                            float64
 2   oldbalanceOrg                     float64
 3   newbalanceOrig                    float64
 4   oldbalanceDest                    float64
 5   newbalanceDest                    float64
 6   isFraud                           int64  
 7   isFlaggedFraud                    int64  
 8   origin_balance_change             float64
 9   destination_balance_change        float64
 10  origin_balance_removed_pct        float64
 11  destination_balance_increase_pct  float64
 12  origin_balance_error              float64
 13  destination_balance_error         float64
 14  high_value_transaction            int64  
 15  type_CASH_OUT                     int64  
 16  type_DEBIT                        in

(6362620, 19)

In [15]:
df_fe.to_csv(
    "../data/paysim_engineered.csv",
    index=False
)

In [16]:
# | Feature                            | Reason                                      
# | ---------------------------------------------------------------------------------- 
# | origin_balance_change              - Measures money removed from sender          
# | destination_balance_change         - Measures money received                     
# | origin_balance_removed_pct        - Percentage of sender's balance removed      
# | destination_balance_increase_pct   - Percentage increase in receiver balance     
# | origin_balance_error               - Detects balance inconsistencies             
# | destination_balance_error          - Detects balance inconsistencies             
# | high_value_transaction            - Flags unusually large transactions          
# | type_*                             - Encodes transaction categories for modeling 
